In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt

from rqfactor import Factor, execute_factor

# ---------- 1. 项目路径：必须从 notebooks/04_report_core_field_audit.ipynb 运行 ----------
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_DIR = NOTEBOOK_DIR.parent
EXPECTED_ROOT_NAME = "a-share-profitability-factor-replication"

if PROJECT_DIR.name != EXPECTED_ROOT_NAME:
    raise RuntimeError(
        "Notebook 工作目录不正确。\n"
        "当前目录：{}\n"
        "期望 Notebook 位于：.../{}/notebooks".format(
            NOTEBOOK_DIR, EXPECTED_ROOT_NAME
        )
    )

TABLE_DIR = PROJECT_DIR / "artifacts" / "tables"
META_DIR = PROJECT_DIR / "artifacts" / "metadata"
FIG_DIR = PROJECT_DIR / "artifacts" / "figures"

for directory in [TABLE_DIR, META_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Notebook 工作目录：", NOTEBOOK_DIR)
print("项目根目录：", PROJECT_DIR)


# ---------- 2. 小样本字段审计：只计算覆盖率，不保存个股数值 ----------
TEST_IDS = [
    "600519.XSHG",  # 贵州茅台
    "000333.XSHE",  # 美的集团
    "600900.XSHG",  # 长江电力
    "601012.XSHG",  # 隆基绿能
]
TEST_START = "2025-06-03"
TEST_END = "2025-06-10"

print("当前环境不提供字段目录函数；将直接查询每个候选字段。")


# ---------- 3. 研报核心公式与候选字段 ----------
SPECS = []

def add_spec(
    factor_key,
    factor_name,
    report_chapter,
    method,
    formula,
    component_key,
    component_name,
    necessity,
    candidates
):
    SPECS.append(
        {
            "factor_key": factor_key,
            "factor_name": factor_name,
            "report_chapter": report_chapter,
            "method": method,
            "formula": formula,
            "component_key": component_key,
            "component_name": component_name,
            "necessity": necessity,
            "candidates": candidates,
        }
    )


# 3.1 稳定 ROE：研报 TTM 版本使用过去 9 期 TTM ROE 波动率
for lag in range(9):
    add_spec(
        factor_key="stable_roe_ttm",
        factor_name="稳定 ROE（TTM）",
        report_chapter="第2章",
        method="direct_inputs",
        formula="zscore(ROE_TTM) + zscore(-std(过去9期ROE_TTM))，等权合成",
        component_key="roe_ttm_lag_{}".format(lag),
        component_name="TTM ROE 滞后 {} 期".format(lag),
        necessity="required",
        candidates=[
            "return_on_equity_weighted_average_ttm_{}".format(lag)
        ],
    )

# 单季度字段名需审计确认；不能确认前不把它当作已复现
for lag in range(12):
    add_spec(
        factor_key="stable_roe_quarterly",
        factor_name="稳定 ROE（单季度）",
        report_chapter="第2章",
        method="candidate_name_audit",
        formula="zscore(单季度ROE) + zscore(-std(过去12期单季度ROE))，等权合成",
        component_key="roe_quarterly_lag_{}".format(lag),
        component_name="单季度 ROE 滞后 {} 期".format(lag),
        necessity="required",
        candidates=[
            "return_on_equity_weighted_average_{}".format(lag),
            "return_on_equity_single_quarter_{}".format(lag),
        ],
    )

# 3.2 ROA：优先用透明重构，直接字段只作为交叉核验
add_spec(
    "roa_ttm_reconstructed",
    "ROA（TTM，重构）",
    "第3章",
    "transparent_reconstruction",
    "归母净利润TTM / 总资产MRQ",
    "roa_net_profit_ttm",
    "归母净利润 TTM",
    "required",
    ["net_profit_parent_company_ttm_0"],
)
add_spec(
    "roa_ttm_reconstructed",
    "ROA（TTM，重构）",
    "第3章",
    "transparent_reconstruction",
    "归母净利润TTM / 总资产MRQ",
    "roa_total_assets_mrq",
    "总资产 MRQ",
    "required",
    ["total_assets_mrq_0"],
)
add_spec(
    "roa_ttm_vendor_reference",
    "ROA（TTM，供应商字段核验）",
    "第3章",
    "vendor_reference",
    "仅用于与重构 ROA 交叉核验，不视为报告口径已复现",
    "roa_vendor_field",
    "供应商 ROA 字段",
    "required",
    ["return_on_asset_ttm", "return_on_assets_ttm"],
)

# 3.3 报告口径 ROIC：NOPLAT / 投入资本
add_spec(
    "roic_ttm_reconstructed",
    "ROIC（TTM，报告口径重构）",
    "第4章",
    "transparent_reconstruction",
    "NOPLAT / (权益 + 有息负债 - 现金及等价物)",
    "roic_ebit_ttm",
    "EBIT TTM",
    "required",
    ["ebit_ttm_0", "ebit_ttm", "ebit"],
)
add_spec(
    "roic_ttm_reconstructed",
    "ROIC（TTM，报告口径重构）",
    "第4章",
    "transparent_reconstruction",
    "NOPLAT / (权益 + 有息负债 - 现金及等价物)",
    "roic_tax_rate",
    "有效税率或所得税输入",
    "required",
    [
        "effective_tax_rate_ttm",
        "income_tax_ttm_0",
        "income_tax_expense_ttm_0",
        "income_tax_expenseTTM",
    ],
)
for component_key, component_name, candidates in [
    ("roic_equity", "总权益 MRQ", ["total_equity_mrq_0"]),
    ("roic_interest_debt", "有息负债", ["interest_bearing_debt"]),
    ("roic_cash", "现金及现金等价物", ["cash_equivalent"]),
]:
    add_spec(
        "roic_ttm_reconstructed",
        "ROIC（TTM，报告口径重构）",
        "第4章",
        "transparent_reconstruction",
        "NOPLAT / (权益 + 有息负债 - 现金及等价物)",
        component_key,
        component_name,
        "required",
        candidates,
    )

add_spec(
    "roic_ttm_vendor_reference",
    "ROIC（TTM，供应商字段核验）",
    "第4章",
    "vendor_reference",
    "仅用于与重构 ROIC 交叉核验，不视为报告口径已复现",
    "roic_vendor_field",
    "供应商 ROIC 字段",
    "required",
    ["return_on_invested_capital_ttm"],
)

# 3.4 RONOA：经营性利润 / 经营性净资产
for component_key, component_name, necessity, candidates in [
    (
        "ronoa_operating_profit",
        "营业利润 TTM",
        "required",
        ["profit_from_operation_ttm_0"],
    ),
    (
        "ronoa_financial_expense",
        "财务费用 TTM",
        "required",
        ["financial_expenseTTM", "financial_expense_ttm_0"],
    ),
    (
        "ronoa_investment_income",
        "投资收益 TTM",
        "required",
        ["investment_income_ttm_0"],
    ),
    (
        "ronoa_fair_value_change",
        "公允价值变动收益 TTM",
        "required",
        ["fair_value_change_income_ttm_0"],
    ),
    (
        "ronoa_equity",
        "总权益 MRQ",
        "required",
        ["total_equity_mrq_0"],
    ),
    (
        "ronoa_interest_debt",
        "有息负债",
        "required",
        ["interest_bearing_debt"],
    ),
    (
        "ronoa_cash",
        "现金及现金等价物",
        "required",
        ["cash_equivalent"],
    ),
    (
        "ronoa_fixed_deposits",
        "定期存款 MRQ",
        "optional",
        ["fixed_deposits_mrq_0"],
    ),
    (
        "ronoa_trading_financial_assets",
        "交易性金融资产 MRQ",
        "optional",
        ["financial_asset_held_for_trading_mrq_0"],
    ),
    (
        "ronoa_available_for_sale_assets",
        "可供出售金融资产 MRQ",
        "optional",
        ["financial_asset_available_for_sale_mrq_0"],
    ),
    (
        "ronoa_hold_to_maturity_assets",
        "持有至到期投资 MRQ",
        "optional",
        ["financial_asset_hold_to_maturity_mrq_0"],
    ),
    (
        "ronoa_dividend_receivable",
        "应收股利 MRQ",
        "optional",
        ["dividend_receivable_mrq_0"],
    ),
]:
    add_spec(
        "ronoa_ttm_reconstructed",
        "RONOA（TTM，重构）",
        "第5章",
        "transparent_reconstruction",
        "(营业利润 + 财务费用 - 投资收益 - 公允价值变动收益) / "
        "(权益 + 有息负债 - 金融性资产)",
        component_key,
        component_name,
        necessity,
        candidates,
    )

# 3.5 FCFFIC：研报 FCFF3 / 投入资本
for factor_key, factor_name, method, cfo_candidates, capex_candidates in [
    (
        "fcffic_ttm_reconstructed",
        "FCFFIC（TTM，重构）",
        "transparent_reconstruction",
        ["cash_flow_from_operating_activities_ttm_0"],
        ["cash_paid_for_asset_ttm_0"],
    ),
    (
        "fcffic_quarterly_reconstructed",
        "FCFFIC（单季度，待审计）",
        "candidate_name_audit",
        [
            "cash_flow_from_operating_activities_0",
            "cash_flow_from_operating_activities",
        ],
        [
            "cash_paid_for_asset_0",
            "cash_paid_for_long_term_assets_0",
        ],
    ),
]:
    add_spec(
        factor_key,
        factor_name,
        "第6章",
        method,
        "(经营现金流 - 资本开支) / (权益 + 有息负债 - 现金及等价物)",
        factor_key + "_cfo",
        "经营现金流",
        "required",
        cfo_candidates,
    )
    add_spec(
        factor_key,
        factor_name,
        "第6章",
        method,
        "(经营现金流 - 资本开支) / (权益 + 有息负债 - 现金及等价物)",
        factor_key + "_capex",
        "资本开支",
        "required",
        capex_candidates,
    )
    for suffix, component_name, candidates in [
        ("equity", "总权益 MRQ", ["total_equity_mrq_0"]),
        ("interest_debt", "有息负债", ["interest_bearing_debt"]),
        ("cash", "现金及现金等价物", ["cash_equivalent"]),
    ]:
        add_spec(
            factor_key,
            factor_name,
            "第6章",
            method,
            "(经营现金流 - 资本开支) / (权益 + 有息负债 - 现金及等价物)",
            factor_key + "_" + suffix,
            component_name,
            "required",
            candidates,
        )

SPEC_DF = pd.DataFrame(SPECS)
print("待审计公式组件数：", len(SPEC_DF))

In [ ]:
def test_one_field(rqfactor_field):
    try:
        panel = execute_factor(
            Factor(rqfactor_field),
            TEST_IDS,
            TEST_START,
            TEST_END,
        )

        valid_observations = int(panel.notna().sum().sum())
        total_observations = int(panel.size)
        coverage = (
            round(valid_observations / total_observations, 4)
            if total_observations > 0
            else 0.0
        )

        return {
            "field_query_success": True,
            "valid_observations": valid_observations,
            "total_observations": total_observations,
            "coverage": coverage,
            "test_result": (
                "available"
                if valid_observations > 0
                else "no_valid_observation"
            ),
        }

    except Exception as error:
        return {
            "field_query_success": False,
            "valid_observations": 0,
            "total_observations": 0,
            "coverage": 0.0,
            "test_result": "query_error: {}".format(str(error)[:160]),
        }


# ---------- 1. 直接查询每一个候选字段 ----------
rows = []

for _, spec in SPEC_DF.iterrows():
    for candidate_rank, rqfactor_field in enumerate(
        spec["candidates"], start=1
    ):
        result = test_one_field(rqfactor_field)

        rows.append(
            {
                "factor_key": spec["factor_key"],
                "factor_name": spec["factor_name"],
                "report_chapter": spec["report_chapter"],
                "method": spec["method"],
                "formula": spec["formula"],
                "component_key": spec["component_key"],
                "component_name": spec["component_name"],
                "necessity": spec["necessity"],
                "candidate_rank": candidate_rank,
                "rqfactor_field": rqfactor_field,
                "test_ids": ",".join(TEST_IDS),
                "test_start": TEST_START,
                "test_end": TEST_END,
                **result,
            }
        )

field_availability = pd.DataFrame(rows)


# ---------- 2. 每个组件挑选第一个查询成功且有值的字段 ----------
usable_fields = (
    field_availability[
        field_availability["field_query_success"]
        & (field_availability["valid_observations"] > 0)
    ]
    .sort_values(["component_key", "candidate_rank"])
    .drop_duplicates("component_key", keep="first")
)

selected_fields = usable_fields[
    [
        "component_key",
        "rqfactor_field",
        "coverage",
        "test_result",
    ]
].rename(
    columns={
        "rqfactor_field": "selected_rqfactor_field",
        "coverage": "selected_coverage",
        "test_result": "selected_test_result",
    }
)

component_map = SPEC_DF.drop(columns=["candidates"]).merge(
    selected_fields,
    on="component_key",
    how="left",
)

component_map["component_ready"] = (
    component_map["selected_rqfactor_field"].notna()
)


# ---------- 3. 汇总每个研报因子的输入就绪状态 ----------
GROUP_COLUMNS = [
    "factor_key",
    "factor_name",
    "report_chapter",
    "method",
    "formula",
]

def summarize_factor(group):
    required = group[group["necessity"] == "required"]
    optional = group[group["necessity"] == "optional"]

    required_count = len(required)
    ready_required = int(required["component_ready"].sum())

    optional_count = len(optional)
    ready_optional = int(optional["component_ready"].sum())

    if required_count == 0:
        readiness_status = "reference_only"
    elif ready_required < required_count:
        readiness_status = "blocked_missing_required_inputs"
    elif optional_count > 0 and ready_optional < optional_count:
        readiness_status = "inputs_ready_proxy_only"
    elif group["method"].iloc[0] == "vendor_reference":
        readiness_status = "vendor_field_available_not_formula_verified"
    else:
        readiness_status = "inputs_ready"

    return pd.Series(
        {
            "required_component_count": required_count,
            "ready_required_component_count": ready_required,
            "optional_component_count": optional_count,
            "ready_optional_component_count": ready_optional,
            "readiness_pct": round(
                100 * ready_required / required_count,
                1,
            ) if required_count else 0.0,
            "readiness_status": readiness_status,
        }
    )

factor_definition_map = (
    component_map.groupby(GROUP_COLUMNS, dropna=False)
    .apply(summarize_factor)
    .reset_index()
    .sort_values(["report_chapter", "factor_key"])
)


# ---------- 4. 保存：不保存任何个股级原始数值 ----------
candidate_query_log = field_availability[
    [
        "factor_key",
        "factor_name",
        "component_key",
        "component_name",
        "necessity",
        "candidate_rank",
        "rqfactor_field",
        "field_query_success",
        "valid_observations",
        "total_observations",
        "coverage",
        "test_result",
    ]
].copy()

field_path = TABLE_DIR / "04_report_core_field_availability.csv"
query_log_path = TABLE_DIR / "04_report_core_candidate_query_log.csv"
definition_path = TABLE_DIR / "04_factor_definition_map.csv"
metadata_path = META_DIR / "04_report_replication_scope.json"
figure_path = FIG_DIR / "04_report_core_coverage.png"

field_availability.to_csv(
    field_path,
    index=False,
    encoding="utf-8-sig",
)

candidate_query_log.to_csv(
    query_log_path,
    index=False,
    encoding="utf-8-sig",
)

factor_definition_map.to_csv(
    definition_path,
    index=False,
    encoding="utf-8-sig",
)

metadata = {
    "project_dir": str(PROJECT_DIR),
    "purpose": "Audit RQFactor inputs for report-core factor replication.",
    "test_ids": TEST_IDS,
    "test_window": {
        "start": TEST_START,
        "end": TEST_END,
    },
    "raw_stock_level_values_saved": False,
    "catalog_access_note": (
        "This RiceQuant environment does not expose "
        "get_all_factor_names. Candidate fields were tested directly."
    ),
    "factor_readiness": json.loads(
        factor_definition_map.to_json(
            orient="records",
            force_ascii=False,
        )
    ),
}

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(
        metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )


# ---------- 5. 图：各因子所需输入的就绪比例 ----------
plot_data = factor_definition_map.sort_values(
    "readiness_pct",
    ascending=True,
).copy()

colors = []

for status in plot_data["readiness_status"]:
    if status in [
        "inputs_ready",
        "vendor_field_available_not_formula_verified",
    ]:
        colors.append("#00A6A6")
    elif status == "inputs_ready_proxy_only":
        colors.append("#F4A261")
    else:
        colors.append("#E76F51")

fig, ax = plt.subplots(figsize=(10, 5))

ax.barh(
    plot_data["factor_key"],
    plot_data["readiness_pct"],
    color=colors,
)

ax.set_xlim(0, 100)
ax.set_xlabel("Required input readiness (%)")
ax.set_title("Report Core Factor Input Audit")
ax.grid(axis="x", alpha=0.25)

for y_position, value in enumerate(plot_data["readiness_pct"]):
    ax.text(
        min(value + 1.5, 96),
        y_position,
        "{:.1f}%".format(value),
        va="center",
        fontsize=9,
    )

fig.tight_layout()
fig.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)


# ---------- 6. 显示结果 ----------
print("\n===== 因子定义与输入就绪状态 =====")
display(factor_definition_map)

print("\n===== 缺失或报错的候选字段 =====")
display(
    field_availability[
        field_availability["test_result"] != "available"
    ][
        [
            "factor_key",
            "component_name",
            "rqfactor_field",
            "candidate_rank",
            "test_result",
        ]
    ]
)

print("\n已保存：", field_path.resolve())
print("已保存：", query_log_path.resolve())
print("已保存：", definition_path.resolve())
print("已保存：", metadata_path.resolve())
print("已保存：", figure_path.resolve())